# Vígil.ia — Validação por classe (vídeos, sem treino)

Mede **acurácia por classe** do detector usando vídeos onde a classe é conhecida
pela **pasta**: cada subpasta tem vídeo(s) só daquela classe (grãos multi-grão,
espalhados, sem se encostar).

O modelo roda em cada vídeo, rastreia cada grão e fecha a classe por **voto temporal**
(ponderado pela confiança). Como a verdade é a pasta, dá pra montar a **matriz de
confusão** e ver exatamente onde ele erra — sem anotar nada, sem treinar, sem salvar.

> Só validação. Não altera nem salva nenhum modelo.
> Estrutura esperada no Drive:
> ```
> <VAL_ROOT>/
>   Intacto/           -> intact
>   Quebrado/          -> broken
>   Imaturo/           -> immature
>   Manchado/          -> spotted
>   Casca danificada/  -> skin-damaged
> ```
> Só as pastas que **tiverem vídeo** entram na validação (as vazias são ignoradas).

## 0. Setup

In [ ]:
!pip -q install "ultralytics==8.4.80"

import torch, ultralytics
ultralytics.checks()
assert torch.cuda.is_available(), 'Sem GPU! (Ambiente de execução -> Alterar tipo -> GPU)'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os, unicodedata
from google.colab import drive
drive.mount('/content/drive')

# --- modelo(s) a validar ---
CHAMPION_PT = '/content/drive/MyDrive/soja_yolo11x_v3.pt'        # campeão atual
ACTIVE_PT   = '/content/drive/MyDrive/soja_yolo11x_active.pt'    # opcional (se existir, valida os 2)
assert os.path.exists(CHAMPION_PT), f'modelo não encontrado: {CHAMPION_PT}'

# --- raiz com UMA subpasta por classe (cada uma com vídeo(s) só daquela classe) ---
# atenção ao acento em "Vídeos". Troque p/ .../Validação quando ela tiver os vídeos.
VAL_ROOT = '/content/drive/MyDrive/Vídeos para treino/Treino'
assert os.path.isdir(VAL_ROOT), (
    f'pasta não encontrada: {VAL_ROOT}\n'
    'confira com:  !ls -R "/content/drive/MyDrive/Vídeos para treino"')

NAMES = ['broken', 'immature', 'intact', 'skin-damaged', 'spotted']
PT = {'broken': 'quebrado', 'immature': 'imaturo', 'intact': 'intacto',
      'skin-damaged': 'casca-dan', 'spotted': 'manchado'}

# casa o nome da pasta (PT/EN, com ou sem acento) -> índice da classe
ALIASES = {0: ['broken', 'quebrad'], 1: ['immature', 'imatur', 'nao maduro'],
           2: ['intact'], 3: ['skin', 'casca', 'ardid', 'danific'],
           4: ['spotted', 'manchad']}

def norm(s):
    return unicodedata.normalize('NFKD', s).encode('ascii', 'ignore').decode().lower()

def class_of(name):
    n = norm(name)
    for idx in range(5):
        if any(norm(k) in n for k in ALIASES[idx]):
            return idx
    return None

VIDEO_EXT = ('.mp4', '.mov', '.avi', '.mkv')
CONF, IMGSZ, IOU = 0.35, 640, 0.5   # mesmos parâmetros da inspeção
print('config ok | VAL_ROOT =', VAL_ROOT)

## 1. Descobrir os vídeos de cada classe

In [ ]:
# varre VAL_ROOT: cada subpasta que casa com uma classe e tem vídeo entra na validação
by_class = {}   # nome da classe -> [caminhos de vídeo]
for entry in sorted(os.listdir(VAL_ROOT)):
    sub = os.path.join(VAL_ROOT, entry)
    if not os.path.isdir(sub):
        continue
    c = class_of(entry)
    if c is None:
        print(f'  (ignorada)  "{entry}" não casa com nenhuma das 5 classes')
        continue
    vids = [os.path.join(sub, f) for f in sorted(os.listdir(sub))
            if f.lower().endswith(VIDEO_EXT)]
    if not vids:
        print(f'  (sem vídeo) "{entry}" -> {NAMES[c]}')
        continue
    by_class[NAMES[c]] = vids
    print(f'  "{entry}" -> {NAMES[c]:12s}: {len(vids)} vídeo(s)')

assert by_class, 'Nenhuma pasta de classe com vídeo em VAL_ROOT — confira a estrutura.'
print('\nclasses que serão validadas:', list(by_class))

## 2. Validar — roda o modelo e vota a classe de cada grão

Cada grão rastreado recebe a classe mais votada (peso = confiança). A verdade é a
pasta de origem. Mede a **acurácia de classificação dos grãos que o modelo detecta**
(grão que ele não detecta não aparece aqui — isso é acurácia de classe, não recall).

In [ ]:
from collections import defaultdict, Counter
from ultralytics import YOLO

def validar(weights):
    model = YOLO(weights)
    y_true, y_pred = [], []
    for true_cls, vids in by_class.items():
        votes = defaultdict(Counter)      # (vídeo, id do grão) -> votos por classe
        for vid in vids:
            for r in model.track(source=vid, imgsz=IMGSZ, iou=IOU, conf=CONF,
                                 agnostic_nms=True, tracker='bytetrack.yaml',
                                 stream=True, verbose=False):
                if r.boxes.id is None:
                    continue
                for tid, c, cf in zip(r.boxes.id.int().tolist(),
                                      r.boxes.cls.int().tolist(),
                                      r.boxes.conf.tolist()):
                    votes[(vid, tid)][NAMES[c]] += cf
        for cnt in votes.values():
            y_true.append(true_cls)
            y_pred.append(cnt.most_common(1)[0][0])
        print(f'  {true_cls:12s}: {sum(1 for t in y_true if t==true_cls)} grãos')
    return y_true, y_pred

MODELS = [('campeão v3', CHAMPION_PT)]
if os.path.exists(ACTIVE_PT):
    MODELS.append(('active (corrigido)', ACTIVE_PT))

results = {}
for tag, w in MODELS:
    print(f'\n>>> validando: {tag}')
    results[tag] = validar(w)
print('\nfeito.')

## 3. Resultados — acurácia por classe + matriz de confusão

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def confusao(y_true, y_pred):
    idx = {c: i for i, c in enumerate(NAMES)}
    M = np.zeros((5, 5), int)
    for t, p in zip(y_true, y_pred):
        M[idx[t], idx[p]] += 1
    return M

for tag, (y_true, y_pred) in results.items():
    M = confusao(y_true, y_pred)
    n = len(y_true)
    acc = 100 * sum(t == p for t, p in zip(y_true, y_pred)) / max(n, 1)
    print(f'\n================  {tag}  ================')
    print(f'grãos avaliados: {n}   |   acurácia geral: {acc:.1f}%\n')
    for i, c in enumerate(NAMES):
        tot = M[i].sum()
        if tot == 0:
            continue   # classe sem vídeo
        certos = M[i, i]
        errs = {NAMES[j]: int(M[i, j]) for j in range(5) if j != i and M[i, j] > 0}
        print(f'  {c:12s}: {certos}/{tot} certos ({100*certos/tot:.1f}%)   '
              f'confundiu com -> {errs if errs else "—"}')

    present = [i for i in range(5) if M[i].sum() > 0]   # só linhas com dado
    fig, ax = plt.subplots(figsize=(5.6, 4.9))
    ax.imshow(M, cmap='Blues')
    ax.set_xticks(range(5)); ax.set_xticklabels([PT[c] for c in NAMES], rotation=45, ha='right')
    ax.set_yticks(range(5)); ax.set_yticklabels([PT[c] for c in NAMES])
    ax.set_xlabel('previsto pelo modelo'); ax.set_ylabel('verdadeiro (pasta)')
    ax.set_title(f'{tag} — matriz de confusão')
    thr = M.max() / 2 if M.max() else 1
    for i in range(5):
        for j in range(5):
            if M[i, j]:
                ax.text(j, i, int(M[i, j]), ha='center', va='center',
                        color='white' if M[i, j] > thr else 'black')
    plt.tight_layout(); plt.show()

print('\nLeitura: a DIAGONAL é acerto. Fora da diagonal = confusão.')
print('Se houver active além do campeão, compare as diagonais: maior = melhor.')